# Adaptive Time-Frequency Generation
Select a GPU runtime, then run the setup, settings, and launch cells. The experiment trains **fixed**, **global**, and **patch** fusion in sequence. Background training lets you rerun the log and result cells while it runs. Saved checkpoint intervals resume automatically.

The default experiment uses ETTh1, length 128, 5000 optimizer steps per control, and seed 42. Results are saved in Google Drive. Setup downloads public repository code directly. For a private repository, it reads a GitHub token from a Colab secret named GITHUB_TOKEN, or a hidden input prompt.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import getpass
import io
import json
import os
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile

drive.mount('/content/drive')
REPOSITORY = 'Zhou-Leheng/adaptive-time-frequency-generation'
request = urllib.request.Request(
    f'https://api.github.com/repos/{REPOSITORY}/zipball/main',
    headers={'Accept': 'application/vnd.github+json'},
)
try:
    response = urllib.request.urlopen(request, timeout=90)
except urllib.error.HTTPError as error:
    if error.code not in (401, 404):
        raise
    try:
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = getpass.getpass('GitHub token with repository contents read access: ')
    request.add_header('Authorization', f'Bearer {github_token}')
    response = urllib.request.urlopen(request, timeout=90)
    del github_token

with response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
del request, response
root = Path('/content/atfg_code')
root.mkdir(exist_ok=True)
for member in archive.infolist():
    target = (root / member.filename).resolve()
    if not target.is_relative_to(root.resolve()):
        raise ValueError('Unexpected archive path')
archive.extractall(root)
CODE = root / archive.namelist()[0].split('/')[0]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')])
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before starting the experiment.'
print('GPU:', torch.cuda.get_device_name(0))
print('Code:', CODE)

In [ ]:
RUN_NAME = 'etth1_seed42'
MODES = ['fixed', 'global', 'patch']
SEED = 42
STEPS = 5000
BATCH_SIZE = 64
OUTPUT = Path('/content/drive/MyDrive/ATFG') / RUN_NAME
LOG = Path('/content') / f'atfg_{RUN_NAME}.log'
PID_RECORD = Path('/content') / f'atfg_{RUN_NAME}_pid.json'
print('Results:', OUTPUT)
print('Log:', LOG)

In [ ]:
from filelock import FileLock
OUTPUT.mkdir(parents=True, exist_ok=True)
# Verify availability before launching. The child holds this lock for the whole run.
with FileLock(str(OUTPUT / '.run.lock'), timeout=0):
    pass
command = [
    'nohup', sys.executable, '-u', str(CODE / 'run.py'),
    '--download', '--data', '/content/ETTh1.csv',
    '--output', str(OUTPUT), '--seed', str(SEED),
    '--steps', str(STEPS), '--batch-size', str(BATCH_SIZE),
    '--modes', *MODES,
]
with LOG.open('a') as handle:
    process = subprocess.Popen(command, cwd=CODE, stdout=handle, stderr=subprocess.STDOUT,
                               stdin=subprocess.DEVNULL, start_new_session=True)
PID_RECORD.write_text(json.dumps({'pid': process.pid, 'script': str(CODE / 'run.py')}))
print('PID:', process.pid)
print('Rerun the next cell to check progress.')

In [ ]:
if LOG.exists():
    print(LOG.read_text(errors='replace')[-8000:])
else:
    print('The log has not been created yet.')

In [ ]:
import csv
from IPython.display import display, Image
summary = OUTPUT / 'summary.csv'
if summary.exists():
    with summary.open() as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        print(row['mode'], 'SWD=', row['swd'], 'PSD=', row['psd_relative_l1'],
              'ACF=', row['acf_mae'], 'delta_W1=', row['delta_w1'])
    for name in ('samples.png', 'router.png'):
        image = OUTPUT / 'patch' / name
        if image.exists():
            display(Image(filename=str(image)))
else:
    print('The first control is still training or sampling. Check the log above.')

To stop gracefully, set STOP = True in the optional cell below and run it. Then rerun the launch cell to resume. A runtime disconnect resumes from the latest saved checkpoint interval. Use a new RUN_NAME when changing experiment settings.

In [ ]:
STOP = False
if STOP:
    import signal
    record = json.loads(PID_RECORD.read_text())
    command_line = Path(f"/proc/{record['pid']}/cmdline")
    if command_line.exists() and record['script'].encode() in command_line.read_bytes().split(b'\x00'):
        os.kill(record['pid'], signal.SIGTERM)
        print('Stop requested. Check the log for the checkpoint confirmation.')
    else:
        print('The recorded training process has already exited.')